# NestedKriging on the Branin 2D function (Python)

This notebook demonstrates **NestedKriging**, a divide-and-conquer Gaussian Process for
large designs: the data are partitioned into $p$ groups, one Kriging submodel is fitted
per group (all sharing a **common prior** after hyperparameter unification), and
predictions are aggregated.

Available aggregations:
- **NK** (default): the optimal aggregation of Rullière, Durrande, Bachoc & Chevalier
  (*Statistics & Computing*, 2018) — itself a kriging predictor: it interpolates the
  design and provides consistent variances;
- **PoE / gPoE / BCM / rBCM**: precision-weighted products of experts (cheaper, but
  no interpolation guarantee).

Fit cost drops from $O(n^3)$ to $O(n^3/p^2)$ per likelihood evaluation.

Steps:
1. Setup
2. Branin function
3. Design of experiments ($n = 800$)
4. Fit `NestedKriging` and inspect the partition
5. Predict on a grid: mean, stdev, RMSE, interpolation check
6. Compare aggregations
7. Warped variant (WarpKriging submodels)


## 0. Installation

Set `REPO_ROOT`, then optionally create a venv and install build requirements
(skip the build cells if pylibkriging is already installed).

In [ ]:
%%bash
REPO_ROOT=$(cd ../.. && pwd)
echo "REPO_ROOT=${REPO_ROOT}"

In [ ]:
%%bash
# Optional: skip if pylibkriging is already installed
set -e
REPO_ROOT=$(cd ../.. && pwd)
VENV_DIR=./venv

# Create venv if needed
if [ ! -d "${VENV_DIR}" ]; then
    python3 -m venv "${VENV_DIR}"
fi
source "${VENV_DIR}/bin/activate"

# Install build requirements
pip install -q \
    -r "${REPO_ROOT}/bindings/Python/pylibkriging/requirements.txt" \
    -r "${REPO_ROOT}/bindings/Python/pylibkriging/dev-requirements.txt"

pip install matplotlib

In [ ]:
%%bash
# Optional: compile libkriging from source if not already built
set -e
REPO_ROOT=$(cd ../.. && pwd)

    cd "${REPO_ROOT}"
    # Point repo-level venv to our local one so loadenv.sh picks it up
    if [ ! -e venv ]; then
        ln -s bindings/Python/venv venv
    fi
    source venv/bin/activate
    
if [ -d "${REPO_ROOT}/build/installed" ]; then
    echo "libkriging already built, skipping build step"
else
    # Force cmake to use the venv python
    EXTRA_CMAKE_OPTIONS="-DPYTHON_EXECUTABLE=$(which python3)" \
        ENABLE_PYTHON_BINDING=on BUILD_TEST=false \
        tools/linux-macos/build.sh
fi


In [ ]:
%%bash
REPO_ROOT=$(cd ../.. && pwd)
# Optional: skip if pylibkriging is already installed
pip install --no-build-isolation ${REPO_ROOT}/bindings/Python/pylibkriging/

## 1. Load pylibkriging

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pylibkriging as lk

print("pylibkriging version:", lk.__version__)

## 2. Branin function

Standard 2D benchmark, rescaled on $[0,1]^2$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def branin(X):
    x1 = X[:, 0] * 15 - 5
    x2 = X[:, 1] * 15
    return (x2 - 5 / (4 * np.pi**2) * x1**2 + 5 / np.pi * x1 - 6)**2 \
           + 10 * (1 - 1 / (8 * np.pi)) * np.cos(x1) + 10

grid_x = np.linspace(0, 1, 50)
G1, G2 = np.meshgrid(grid_x, grid_x)
grid_pts = np.column_stack([G1.ravel(), G2.ravel()])
z_true = branin(grid_pts).reshape(50, 50)

plt.contourf(G1, G2, z_true, 20)
plt.colorbar(); plt.title("True Branin"); plt.show()

## 3. Design of experiments

$n = 800$ points: a full kriging factorizes one $800\times800$ matrix per likelihood
evaluation; NestedKriging with $p=8$ factorizes eight $100\times100$ blocks.

In [ ]:
rng = np.random.default_rng(123)
n = 800
X = rng.uniform(size=(n, 2))
y = branin(X)
print("design:", X.shape)

## 4. Fit NestedKriging (NK aggregation, 8 groups)

In [ ]:
nk = lk.NestedKriging(y, X, "matern5_2", 8)  # aggregation="NK", partition="kmeans" by default
print(nk.summary())

In [ ]:
plt.figure(figsize=(5, 5))
for g, idx in enumerate(nk.groups()):
    idx = idx.astype(int)
    plt.scatter(X[idx, 0], X[idx, 1], s=8, label=f"group {g} ({len(idx)})")
plt.legend(fontsize=7); plt.title("k-means partition of the design"); plt.show()

## 5. Prediction on the grid

In [ ]:
mean, stdev = nk.predict(grid_pts, True)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
c0 = ax[0].contourf(G1, G2, mean.reshape(50, 50), 20)
fig.colorbar(c0, ax=ax[0]); ax[0].set_title("NK mean")
c1 = ax[1].contourf(G1, G2, stdev.reshape(50, 50), 20)
fig.colorbar(c1, ax=ax[1]); ax[1].set_title("NK stdev")
plt.show()

print("RMSE vs true Branin :", np.sqrt(np.mean((mean - z_true.ravel())**2)))
m_at_X, _ = nk.predict(X, True)
print("max |mean - y| at design points (interpolation):", np.abs(m_at_X - y).max())

## 6. Comparing aggregations

In [ ]:
for agg in ["NK", "gPoE", "rBCM", "PoE", "BCM"]:
    k = lk.NestedKriging(y, X, "matern5_2", 8, aggregation=agg)
    m, s = k.predict(grid_pts, True)
    print(f"{agg:5s}  RMSE = {np.sqrt(np.mean((m - z_true.ravel())**2)):.4f}")

## 7. Warped variant

With `warping`, submodels are `WarpKriging` sharing a common warped prior
$\sigma^2 k(\Phi(x), \Phi(x'))$.

In [ ]:
# (theta, warp) are estimated by a single reference fit on a global subsample:
# here 200 points instead of min(n, 1000) — much faster, same aggregation quality
nkw = lk.NestedKriging("gauss")
nkw.set_warp_subsample(200)
nkw.fit(y, X, 8, warping=["kumaraswamy", "kumaraswamy"])
mw, sw = nkw.predict(grid_pts, True)
print("warping      :", nkw.warping())
print("RMSE (warped):", np.sqrt(np.mean((mw - z_true.ravel())**2)))

## Notes

- The NK aggregation **interpolates** whatever the hyperparameters: it is a property of
  the aggregation itself, not of the fit.
- NK requires a **constant trend**; use the PoE family for other trends.
- Memory/speed of the NK prediction can be tuned with `set_predict_chunk`.
- In the warped case, the prior hyperparameters $(\theta, \text{warp})$ are estimated by a
  **single reference fit on a global subsample** (default 1000 points, see
  `set_warp_subsample`), then submodels are fitted in closed form (`optim="none"`).
